---
layout: post
codemirror: true
title: "Python 3.05: Boolean Expressions HW"
categories: [HW]
lesson_language: Python
lesson_topic: Boolean-Expressions HW
lesson_part: interactive
lesson_type: lesson
permalink: /python/boolean-hw
author: pkdev0101
---

## Popcorn Hack 1: Check One SFI Part Record

**Prediction:** every field is filled in and "Auto Racing" is in the accepted list, so all four checks and the combined result should be `True`. Changing the category to "Street Car" should flip only the category check, which makes the whole record `False`.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Check SFI Part Fields

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]

# Each relational expression evaluates to True or False on its own.
# .strip() removes spaces first, so a name of "   " doesn't count as filled in.
has_product_name = part["product_name"].strip() != ""
# `in` checks membership, so this is True only for an accepted racing category.
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"].strip() != ""
has_effective_date = part["effective_date"].strip() != ""

print("Has product name:", has_product_name)
print("Valid racing category:", valid_category)
print("Has spec number:", has_spec_number)
print("Has effective date:", has_effective_date)

# "Complete" means EVERY requirement passes, so the checks are joined with `and`.
record_complete = has_product_name and valid_category and has_spec_number and has_effective_date
print("Record complete:", record_complete)

# Changed input: the same checks on a Street Car record. Only the category check flips,
# and because `and` needs every part to be True, the whole record becomes incomplete.
part["category"] = "Street Car"
valid_category = part["category"] in valid_categories
print("Changed category to Street Car -> valid category:", valid_category)
print("Changed category to Street Car -> record complete:",
      has_product_name and valid_category and has_spec_number and has_effective_date)

**Actual output:**

```text
Has product name: True
Valid racing category: True
Has spec number: True
Has effective date: True
Record complete: True
Changed category to Street Car -> valid category: False
Changed category to Street Car -> record complete: False
```

The prediction matched. Because the record is "complete" only when every requirement passes, the checks are joined with `and`, so one `False` makes the whole result `False`.

## Popcorn Hack 2: Backend Create Rule

**Prediction:** spec `1.1` is already stored, so the original record should be rejected (`False`). Changing the spec to `1.3`, which isn't stored, should allow it (`True`).

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Backend Create Rule

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

def can_create(part):
    # Small named checks first, then one combined rule. This makes it easy to see which check failed.
    has_fields = (
        part["product_name"].strip() != ""
        and part["spec_number"].strip() != ""
        and part["effective_date"].strip() != ""
    )
    valid_category = part["category"] in valid_categories
    # `not` flips the duplicate check: a record can be created only if its spec is NOT already stored.
    is_new_spec = not (part["spec_number"] in existing_spec_numbers)
    return has_fields and valid_category and is_new_spec

# Rejected case: spec 1.1 is already stored, so is_new_spec is False and `and` makes the result False.
duplicate_part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}
print("Duplicate spec 1.1 -> create record:", can_create(duplicate_part))

# Valid case: same record with a spec number that isn't stored yet, so every check is True.
new_part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.3",
    "effective_date": "Nov. 9, 2001"
}
print("New spec 1.3 -> create record:", can_create(new_part))

**Actual output:**

```text
Duplicate spec 1.1 -> create record: False
New spec 1.3 -> create record: True
```

**Valid case:** spec 1.3 passes every check. **Rejected case:** spec 1.1 fails only the duplicate check, and since every check is joined with `and`, that one failure rejects the record. `not` is what turns "is a duplicate" into "is new."

## Popcorn Hack 3: SFI Search Filter

**Prediction:** query `1.2` should match only the Multiple Disc Clutch record by spec number. For a name search, I added a Street Car record to test the filter: "flywheel" appears in three names, but the Street Car record should be filtered out. "brake" should match nothing.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - SFI Search Filter

query = "1.2"

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Street Flywheel Kit",
        "category": "Street Car",
        "spec_number": "9.9"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]

def search(query, records):
    results = []
    for record in records:
        # `or`: a record matches if EITHER the name contains the query OR the spec number is an exact match.
        # Lowercasing both sides makes the name search ignore capitalization.
        matches_query = (
            query.lower() in record["product_name"].lower()
            or query == record["spec_number"]
        )
        # `and`: a match only counts if the category is accepted too.
        # The parentheses make the `or` run first, so the category filter applies to both kinds of match.
        if matches_query and record["category"] in valid_categories:
            results.append(record["product_name"] + " (" + record["spec_number"] + ")")
    return results

print("Search '1.2':", search(query, records))
# Name match: three records contain "flywheel", but the Street Car one is filtered out by `and`.
print("Search 'flywheel':", search("flywheel", records))
# Nonmatch: no name contains "brake" and no spec equals it, so the list is empty.
print("Search 'brake':", search("brake", records))

**Actual output:**

```text
Search '1.2': ['Multiple Disc Clutch Assemblies (1.2)']
Search 'flywheel': ['Replacement Flywheels and Clutch Assemblies (1.1)', 'Racing Flywheel Record (2.1)']
Search 'brake': []
```

The rule is `(name matches OR spec matches) AND category accepted`. The parentheses matter: without them, Python evaluates `and` before `or`, so a spec-number match could skip the category filter.

## MCQ 3.05

**Question 1:** `7 >= 7` evaluates to **A. True**, because `>=` is true when the values are equal.

MCQ 3.05: 4/4

## Homework Hack: SFI Backend Validator

**Rule:** a record is accepted only if it has a non-empty product name, an accepted category, a non-empty spec number, an effective date, **and** a spec number that is **not** already stored.

**Predictions for the supplied cases:**
1. Multiple Disc Clutch Assemblies (spec 1.2, Auto Racing): **accepted**, since every rule passes.
2. Replacement Flywheels (Street Car): **rejected**, because the category is not accepted.
3. Existing Flywheel Record (spec 1.1): **rejected**, because it's a duplicate spec number.

I also added two edge cases (a name made only of spaces, and a missing date) and a reusability test that resubmits the first record after it was accepted.

In [ ]:
# CODE_RUNNER: Homework Hack - SFI Backend Validator

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

test_records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Existing Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    },
    # Extra edge cases I added: a blank-looking name and a missing date.
    {
        "product_name": "   ",
        "category": "Drag Racing",
        "spec_number": "4.4",
        "effective_date": "Mar. 3, 2026"
    },
    {
        "product_name": "Clutch Cover",
        "category": "Auto Racing",
        "spec_number": "5.5",
        "effective_date": ""
    }
]

def validate_record(record, existing_specs, categories):
    # One named Boolean per rule, so each requirement can be checked on its own.
    has_name = record["product_name"].strip() != ""
    accepted_category = record["category"] in categories
    has_spec = record["spec_number"].strip() != ""
    has_date = record["effective_date"].strip() != ""
    is_duplicate = record["spec_number"] in existing_specs

    # The written rule says ALL five must hold, so they are joined with `and`,
    # and `not` reverses the duplicate check because a duplicate must be rejected.
    is_valid = has_name and accepted_category and has_spec and has_date and not is_duplicate

    # Collect the reasons so a rejection explains exactly which rule failed.
    reasons = []
    if not has_name:
        reasons.append("missing product name")
    if not accepted_category:
        reasons.append("category not accepted")
    if not has_spec:
        reasons.append("missing spec number")
    if not has_date:
        reasons.append("missing effective date")
    if is_duplicate:
        reasons.append("duplicate spec number")
    return is_valid, reasons

for record in test_records:
    is_valid, reasons = validate_record(record, existing_spec_numbers, valid_categories)
    label = record["product_name"].strip() or "(no name)"
    if is_valid:
        # An accepted record is stored, so its spec number now counts as existing.
        existing_spec_numbers.append(record["spec_number"])
        print("ACCEPTED:", label, "| spec", record["spec_number"])
    else:
        print("REJECTED:", label, "| spec", record["spec_number"], "|", ", ".join(reasons))

# Reusability test: submitting the first record again is now a duplicate,
# because the validator stored spec 1.2 when it was accepted above.
is_valid, reasons = validate_record(test_records[0], existing_spec_numbers, valid_categories)
print("Resubmit spec 1.2 ->", "ACCEPTED" if is_valid else "REJECTED: " + ", ".join(reasons))
print("Stored spec numbers:", existing_spec_numbers)

**Actual output:**

```text
ACCEPTED: Multiple Disc Clutch Assemblies | spec 1.2
REJECTED: Replacement Flywheels | spec 3.1 | category not accepted
REJECTED: Existing Flywheel Record | spec 1.1 | duplicate spec number
REJECTED: (no name) | spec 4.4 | missing product name
REJECTED: Clutch Cover | spec 5.5 | missing effective date
Resubmit spec 1.2 -> REJECTED: duplicate spec number
Stored spec numbers: ['1.1', '2.1', '1.2']
```

**How the Boolean logic works:** Each requirement is its own named Boolean (`has_name`, `accepted_category`, `has_spec`, `has_date`, `is_duplicate`). The written rule says *all* requirements must pass, so they're combined with `and`, and `not is_duplicate` handles the "must not already exist" rule. Using `.strip()` before `!= ""` means a name of only spaces still counts as missing. When a record is accepted, its spec number is added to the stored list, so submitting the same record again is correctly rejected as a duplicate. That shows the validator is reusable, not just correct for one fixed list.

**Same rule in College Board pseudocode** (for record 3, the duplicate):

```text
IF (name ≠ "" AND category IN valid_categories AND spec ≠ "" AND date ≠ "" AND NOT (spec IN existing_specs))
{
    DISPLAY("ACCEPTED")
}
ELSE
{
    DISPLAY("REJECTED")
}
```

Prediction: spec `1.1` is in `existing_specs`, so `NOT (...)` is `false`, the whole `AND` is `false`, and it displays REJECTED. The Python validator made the same decision. (College Board uses `=` and `≠` for comparison, while Python uses `==` and `!=`.)